# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signals

- `age_days`: number of days from the last update to the decision date.
- `decline_ratio`: `(H1 − H2) / H1`, the ratio of the impressions drop between the two halves of month 3.


- `age_norm` = `min(age_days / 365, 1)`
- `decline_norm` = `min(max(decline_ratio, 0), 1)`

### Score

`score = w1 × age_norm + w2 × decline_norm`

In [5]:
import duckdb, pandas as pd
from getpass import getpass
from huggingface_hub import login

TOKEN = getpass("HF token: ")
login(token=TOKEN)

con = duckdb.connect()
con.execute(f"CREATE SECRET hf (TYPE huggingface, TOKEN '{TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
SRC_DAILY   = f"{BASE}/fact_content_daily_performance/**/*.parquet"
SRC_CLIENTS = f"{BASE}/dim_clients.parquet"
SRC_CONTENT = f"{BASE}/dim_content.parquet"

START, END  = "2026-03-01", "2026-03-31"
MIN_IMPR_M3 = 100   # الحد الأدنى للـ impressions في شهر 3

stale = con.execute(f"""
WITH clients AS (
    SELECT client_hash_id, CAST(gsc_data_start AS DATE) AS gsc_data_start
    FROM read_parquet('{SRC_CLIENTS}')
),
daily AS (
    SELECT d.content_hash_id, d.gsc_impressions
    FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
    JOIN clients c USING (client_hash_id)
    WHERE d.month = '2026-03'
      AND d.report_date BETWEEN DATE '{START}' AND DATE '{END}'
      AND d.report_date >= c.gsc_data_start
),
eligible AS (
    SELECT content_hash_id
    FROM daily
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) >= {MIN_IMPR_M3}
)
SELECT k.content_hash_id,
       date_diff(
           'day',
           CASE
               WHEN k.content_updated_date IS NOT NULL
                AND CAST(k.content_updated_date AS DATE) <= DATE '{END}'
               THEN CAST(k.content_updated_date AS DATE)
               ELSE CAST(k.content_created_date AS DATE)
           END,
           DATE '{END}'
       ) AS stale_days
FROM read_parquet('{SRC_CONTENT}') k
JOIN eligible e USING (content_hash_id)
""").df()


HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
stale

,content_hash_id,stale_days
0,content_04c67f3541177192,34
1,content_0f30e04e709c7b5d,34
2,content_1207efddce873942,175
3,content_167472cd0802a8f3,34
4,content_27f8100281413b37,34
...,...,...
101436,content_ff22937a21a14ca6,277
101437,content_0fa6a06226678efb,13
101438,content_208fef584ee9bbee,13
101439,content_2cea81d54b2fe94f,13


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.